In [1]:
# Install necessary libraries
!pip install torch numpy pandas scikit-learn mealpy

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Load data
train_path = r"E:\JN_Minor project\UNSW_NB15\UNSW_NB15\UNSW_NB15_training-set.csv"
test_path = r"E:\JN_Minor project\UNSW_NB15\UNSW_NB15\UNSW_NB15_testing-set.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Encode label column for binary classification
df_train['label'] = df_train['label'].apply(lambda x: 0 if x == 'Normal' else 1)
df_test['label'] = df_test['label'].apply(lambda x: 0 if x == 'Normal' else 1)

# Combine train and test to encode categories consistently
df_train['is_train'] = 1
df_test['is_train'] = 0
combined = pd.concat([df_train, df_test])

# One-hot encode all categorical columns
categorical_cols = combined.select_dtypes(include='object').columns.tolist()
combined_encoded = pd.get_dummies(combined, columns=categorical_cols)

# Separate back train and test
df_train = combined_encoded[combined_encoded['is_train'] == 1].drop(columns=['is_train'])
df_test = combined_encoded[combined_encoded['is_train'] == 0].drop(columns=['is_train'])

# Separate features and target
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']
X_test = df_test.drop(columns=['label'])
y_test = df_test['label']

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define your Neural Network model
class EnhancedNet(nn.Module):
    def __init__(self, input_dim):
        super(EnhancedNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 2)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)

# Fixed hyperparameters (set manually instead of tuning)
fixed_shots = 10
fixed_task_lr = 0.01
fixed_meta_lr = 0.001

# Function to train the model once
def maml_train_once(model, X, y, shots, task_lr, meta_lr, num_tasks=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=meta_lr)
    loss_fn = nn.CrossEntropyLoss()
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.long)
    for task in range(num_tasks):
        indices = np.random.choice(len(X_tensor), shots * 2, replace=False)
        support_idx, query_idx = indices[:shots], indices[shots:]
        support_x, support_y = X_tensor[support_idx], y_tensor[support_idx]
        query_x, query_y = X_tensor[query_idx], y_tensor[query_idx]

        # Inner loop
        temp_model = EnhancedNet(X.shape[1])
        temp_model.load_state_dict(model.state_dict())
        temp_optimizer = torch.optim.SGD(temp_model.parameters(), lr=task_lr)
        temp_optimizer.zero_grad()
        loss = loss_fn(temp_model(support_x), support_y)
        loss.backward()
        temp_optimizer.step()

        # Outer loop update
        optimizer.zero_grad()
        outer_loss = loss_fn(temp_model(query_x), query_y)
        outer_loss.backward()
        optimizer.step()

    return model

# Train the model with fixed hyperparameters
model = EnhancedNet(X_train.shape[1])
model = maml_train_once(model, X_train, y_train, shots=fixed_shots, task_lr=fixed_task_lr, meta_lr=fixed_meta_lr, num_tasks=10)

# Evaluate the model
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_pred = model(X_test_tensor).argmax(dim=1).numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.5700578147014527
Precision: 1.0
Recall: 0.5700578147014527
F1 Score: 0.7261615583370724
